In [2]:
"""
Pipeline multi-metodo de deteccion de periodo (LS + CE + PDM)
Lee HDF5 (singlemode_variable_stars_sample.h5) y/o FITS (VVV_Sample.fits)
con seleccion interactiva.
"""

import io
import csv
import time as pytime
import warnings
import traceback
from pathlib import Path
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed
)
import multiprocessing as mp

import numpy as np
import h5py
from astropy.io import fits
from astropy.timeseries import LombScargle

import matplotlib
matplotlib.use('Agg')
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg


# =============================================================================
# CONFIGURACION
# =============================================================================

DATA_DIR    = Path(r"C:\Users\santi\Downloads\Lab 5")
H5_PATH     = DATA_DIR / "singlemode_variable_stars_sample.h5"
FITS_PATH   = DATA_DIR / "VVV_Sample.fits"
OUTPUT_DIR  = Path("./periods_output")

# Que fuente procesar:
#   'ask'  -> preguntar al usuario al iniciar (por defecto)
#   'h5'   -> solo HDF5
#   'fits' -> solo FITS
#   'both' -> ambos (sin preguntar)
SOURCE_MODE = 'ask'

# Columnas del HDF5
H5_COL_TIME = 'RHJD'
H5_COL_MAG  = 'mag'
H5_COL_ERR  = 'mag_err'

# Campos del FITS VVV
FITS_FIELDS = ('B278', 'B279')

# --- Lomb-Scargle ---
PERIOD_MIN        = 0.05
PERIOD_MAX_FACTOR = 1.0 / 3.0
SAMPLES_PER_PEAK  = 10
MIN_OBSERVATIONS  = 30
FAP_THRESHOLD     = 0.01

ALIAS_PERIODS = np.array([1.0, 0.9973, 0.5, 1/3.0, 0.25, 29.5, 365.25])
ALIAS_TOL     = 0.01

SNR_MIN           = 2.0

TOP_K_LS_PEAKS    = 5
REFINE_HALFWIDTH  = 0.01
REFINE_N_POINTS   = 400
CE_PHASE_BINS     = 10
CE_MAG_BINS       = 5
PDM_N_BINS        = 10

CONSENSUS_RTOL    = 0.01

N_TOP_PLOTS       = 30
PLOT_DPI          = 100

N_WORKERS         = None
EXECUTOR_MODE     = 'auto'  # 'auto' / 'thread' / 'process'


# =============================================================================
# ENTORNO
# =============================================================================

def _in_notebook():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None:
            return False
        return type(ip).__name__ == 'ZMQInteractiveShell' or \
               'ipykernel' in str(type(ip)).lower()
    except Exception:
        return False


def _select_executor_class():
    if EXECUTOR_MODE == 'thread':
        return ThreadPoolExecutor, 'ThreadPoolExecutor (forzado)'
    if EXECUTOR_MODE == 'process':
        return ProcessPoolExecutor, 'ProcessPoolExecutor (forzado)'
    if _in_notebook():
        return ThreadPoolExecutor, 'ThreadPoolExecutor (notebook)'
    return ProcessPoolExecutor, 'ProcessPoolExecutor'


# =============================================================================
# SELECCION DE FUENTE
# =============================================================================

def prompt_source_selection():
    """Pregunta interactivamente que fuente procesar."""
    h5_exists   = H5_PATH.exists()
    fits_exists = FITS_PATH.exists()

    print("\n" + "=" * 60)
    print("Seleccion de fuente de datos")
    print("=" * 60)
    print(f"  [1] HDF5  : {H5_PATH.name:40s}  {'OK' if h5_exists else 'NO ENCONTRADO'}")
    print(f"  [2] FITS  : {FITS_PATH.name:40s}  {'OK' if fits_exists else 'NO ENCONTRADO'}")
    print(f"  [3] Ambos")
    print("=" * 60)

    if not h5_exists and not fits_exists:
        raise FileNotFoundError(
            f"No se encontro ninguno de los archivos en {DATA_DIR}. "
            f"Ajusta DATA_DIR o los paths."
        )

    while True:
        try:
            choice = input("Seleccion [1/2/3]: ").strip()
        except EOFError:
            choice = '3' if (h5_exists and fits_exists) else ('1' if h5_exists else '2')
            print(f"(stdin vacio -> default: {choice})")

        if choice == '1':
            if not h5_exists:
                print(f"  HDF5 no existe en {H5_PATH}. Elige otra opcion.")
                continue
            return ['h5']
        if choice == '2':
            if not fits_exists:
                print(f"  FITS no existe en {FITS_PATH}. Elige otra opcion.")
                continue
            return ['fits']
        if choice == '3':
            sources = []
            if h5_exists:   sources.append('h5')
            if fits_exists: sources.append('fits')
            if len(sources) == 1:
                print(f"  Solo disponible: {sources[0].upper()}. Usando solo esa.")
            return sources
        print("  Entrada invalida. Escribe 1, 2 o 3.")


def resolve_sources():
    mode = SOURCE_MODE.lower()
    if mode == 'ask':
        return prompt_source_selection()
    if mode == 'h5':
        if not H5_PATH.exists():
            raise FileNotFoundError(f"HDF5 no encontrado: {H5_PATH}")
        return ['h5']
    if mode == 'fits':
        if not FITS_PATH.exists():
            raise FileNotFoundError(f"FITS no encontrado: {FITS_PATH}")
        return ['fits']
    if mode == 'both':
        srcs = []
        if H5_PATH.exists():   srcs.append('h5')
        if FITS_PATH.exists(): srcs.append('fits')
        if not srcs:
            raise FileNotFoundError(f"Ninguno de los archivos existe en {DATA_DIR}")
        return srcs
    raise ValueError(f"SOURCE_MODE invalido: {SOURCE_MODE!r}")


# =============================================================================
# LOADERS
# =============================================================================

def load_h5_stars(h5_path):
    """Devuelve lista de (source, name, t, m, e) con source='h5'."""
    stars = []
    with h5py.File(h5_path, 'r') as f:
        for name in f['/'].keys():
            rec = f[name][:]
            t = np.asarray(rec[H5_COL_TIME], dtype=np.float64)
            m = np.asarray(rec[H5_COL_MAG],  dtype=np.float64)
            e = np.asarray(rec[H5_COL_ERR],  dtype=np.float64)
            mask = ~(np.isnan(t) | np.isnan(m) | np.isnan(e))
            stars.append(('h5', str(name), t[mask], m[mask], e[mask]))
    return stars


def load_fits_stars(fits_path):
    """
    Lee VVV_Sample.fits. Devuelve lista de (source, name, t, m, e)
    con source='fits_B278' o 'fits_B279' y name=f'{field}_{id}'.
    """
    stars = []
    with fits.open(fits_path, memmap=False) as hdul:
        for field in FITS_FIELDS:
            try:
                ids = np.array(hdul[f'ID_{field}'].data)
                mag = np.array(hdul[f'KS_{field}'].data,     dtype=np.float64)
                err = np.array(hdul[f'KS_ERR_{field}'].data, dtype=np.float64)
                mjd = np.array(hdul[f'KS_MJD_{field}'].data, dtype=np.float64)
            except KeyError as ex:
                print(f"  [WARN] HDU no encontrada para {field}: {ex}")
                continue

            n_obj = len(ids)
            if mag.ndim == 2 and mag.shape[0] != n_obj and mag.shape[1] == n_obj:
                mag = mag.T
            if err.ndim == 2 and err.shape[0] != n_obj and err.shape[1] == n_obj:
                err = err.T
            if mjd.ndim == 2 and mjd.shape[0] != n_obj and mjd.shape[1] == n_obj:
                mjd = mjd.T

            tag = f'fits_{field}'
            for i, obj_id in enumerate(ids):
                t = mjd[i]; mm = mag[i]; ee = err[i]
                mask = ~(np.isnan(t) | np.isnan(mm) | np.isnan(ee))
                stars.append((tag, f'{field}_{int(obj_id)}',
                              t[mask], mm[mask], ee[mask]))
    return stars


def load_selected(sources):
    all_stars = []
    if 'h5' in sources:
        print(f"\nLeyendo HDF5: {H5_PATH}")
        s = load_h5_stars(H5_PATH)
        print(f"  {len(s)} estrellas.")
        all_stars.extend(s)
    if 'fits' in sources:
        print(f"\nLeyendo FITS: {FITS_PATH}")
        s = load_fits_stars(FITS_PATH)
        print(f"  {len(s)} estrellas (B278 + B279).")
        all_stars.extend(s)
    return all_stars


# =============================================================================
# RUIDO
# =============================================================================

def noise_check(m, e):
    var_total = float(np.var(m, ddof=1))
    var_noise = float(np.mean(e ** 2))
    var_sig   = var_total - var_noise
    if var_sig <= 0:
        return 0.0, 0.0, False
    amp_int = float(np.sqrt(var_sig))
    err_med = float(np.median(e))
    snr = amp_int / err_med if err_med > 0 else 0.0
    return snr, amp_int, snr >= SNR_MIN


# =============================================================================
# LOMB-SCARGLE
# =============================================================================

def run_ls(t, m, e):
    T = t.max() - t.min()
    period_max = T * PERIOD_MAX_FACTOR
    ls = LombScargle(t, m, e, fit_mean=True, center_data=True,
                     normalization='standard')
    frequency, power = ls.autopower(
        minimum_frequency=1.0 / period_max,
        maximum_frequency=1.0 / PERIOD_MIN,
        samples_per_peak=SAMPLES_PER_PEAK,
    )
    return ls, frequency, power


def alias_mask_vec(frequency):
    period = 1.0 / frequency
    alias_arr = np.concatenate([ALIAS_PERIODS, ALIAS_PERIODS / 2.0])
    rel_diff = np.abs(period[:, None] - alias_arr[None, :]) / alias_arr[None, :]
    return ~np.any(rel_diff < ALIAS_TOL, axis=1)


def top_k_peaks(frequency, power, k, min_separation=0.05):
    periods = 1.0 / frequency
    order = np.argsort(power)[::-1]
    peaks = []
    for i in order:
        P_i = periods[i]
        if all(abs(P_i - P_j) / P_j > min_separation for P_j, _ in peaks):
            peaks.append((float(P_i), float(power[i])))
            if len(peaks) >= k:
                break
    return peaks


# =============================================================================
# CE y PDM
# =============================================================================

def conditional_entropy_batch(t, m, periods,
                               n_phase=CE_PHASE_BINS,
                               n_mag=CE_MAG_BINS):
    N = len(t)
    m_min, m_max = m.min(), m.max()
    if m_max - m_min < 1e-12:
        return np.full(len(periods), np.nan)
    m_norm = (m - m_min) / (m_max - m_min)
    m_bin  = np.clip((m_norm * n_mag).astype(np.int64), 0, n_mag - 1)

    H_out = np.empty(len(periods), dtype=np.float64)
    n_cells = n_phase * n_mag

    for k, P in enumerate(periods):
        if P <= 0:
            H_out[k] = np.inf; continue
        phase   = (t / P) % 1.0
        phi_bin = np.clip((phase * n_phase).astype(np.int64), 0, n_phase - 1)
        idx = phi_bin * n_mag + m_bin
        counts = np.bincount(idx, minlength=n_cells).reshape(n_phase, n_mag)
        p_ij = counts / N
        p_phi = p_ij.sum(axis=1, keepdims=True)
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where(p_phi > 0, p_ij / p_phi, 0.0)
            log_r = np.where(ratio > 0, np.log(ratio), 0.0)
        H_out[k] = -np.sum(p_ij * log_r)
    return H_out


def pdm_batch(t, m, periods, n_bins=PDM_N_BINS):
    N = len(t)
    sigma2_tot = float(np.var(m, ddof=1))
    if sigma2_tot < 1e-12:
        return np.full(len(periods), np.nan)

    theta_out = np.empty(len(periods), dtype=np.float64)
    for k, P in enumerate(periods):
        if P <= 0:
            theta_out[k] = np.inf; continue
        phase   = (t / P) % 1.0
        phi_bin = np.clip((phase * n_bins).astype(np.int64), 0, n_bins - 1)

        sum_m  = np.bincount(phi_bin, weights=m,     minlength=n_bins)
        sum_m2 = np.bincount(phi_bin, weights=m * m, minlength=n_bins)
        counts = np.bincount(phi_bin,                 minlength=n_bins).astype(np.float64)

        valid = counts > 1
        if not np.any(valid):
            theta_out[k] = np.nan; continue
        ssq = sum_m2[valid] - (sum_m[valid] ** 2) / counts[valid]
        s2  = ssq.sum() / (counts[valid].sum() - valid.sum())
        theta_out[k] = s2 / sigma2_tot
    return theta_out


def refine_around_peaks(t, m, peaks, halfwidth=REFINE_HALFWIDTH,
                         n_points=REFINE_N_POINTS):
    all_P, all_CE, all_PDM = [], [], []
    for P_peak, _ in peaks:
        grid = np.linspace(P_peak * (1 - halfwidth),
                           P_peak * (1 + halfwidth), n_points)
        all_P.append(grid)
        all_CE.append(conditional_entropy_batch(t, m, grid))
        all_PDM.append(pdm_batch(t, m, grid))

    P_all   = np.concatenate(all_P)
    CE_all  = np.concatenate(all_CE)
    PDM_all = np.concatenate(all_PDM)

    i_ce  = int(np.nanargmin(CE_all))
    i_pdm = int(np.nanargmin(PDM_all))
    return dict(P_CE=float(P_all[i_ce]), CE_min=float(CE_all[i_ce]),
                P_PDM=float(P_all[i_pdm]), PDM_min=float(PDM_all[i_pdm]))


# =============================================================================
# CONSENSO
# =============================================================================

def consensus_period(P_LS, P_CE, P_PDM, rtol=CONSENSUS_RTOL):
    def close(a, b): return abs(a - b) / b < rtol
    c_ls_ce  = close(P_LS, P_CE)
    c_ls_pdm = close(P_LS, P_PDM)
    c_ce_pdm = close(P_CE, P_PDM)
    if c_ls_ce and c_ls_pdm and c_ce_pdm:
        return (P_LS + P_CE + P_PDM) / 3.0, 'LS+CE+PDM', 3
    if c_ce_pdm: return (P_CE + P_PDM) / 2.0, 'CE+PDM', 2
    if c_ls_ce:  return P_CE,  'LS+CE', 2
    if c_ls_pdm: return P_PDM, 'LS+PDM', 2
    return P_CE, 'CE-only', 1


# =============================================================================
# PIPELINE POR ESTRELLA
# =============================================================================

def _analyze_star(args):
    source, name, t, m, e = args
    base = dict(
        source=source, name=name, n_obs=int(len(t)),
        valid=False, error=None,
        baseline=np.nan, mag_mean=np.nan, mag_std=np.nan, err_median=np.nan,
        snr=np.nan, amp_intrinsic=np.nan,
        P_LS=np.nan, power_LS=np.nan, fap_LS=np.nan,
        P_LS_raw=np.nan, alias_flag=False,
        P_CE=np.nan, CE_min=np.nan,
        P_PDM=np.nan, PDM_min=np.nan,
        P_final=np.nan, agreement=None, n_agree=0,
        significant=False,
    )
    try:
        if len(t) < MIN_OBSERVATIONS:
            base['error'] = f'n_obs<{MIN_OBSERVATIONS}'; return base

        T = t.max() - t.min()
        if T <= 0:
            base['error'] = 'baseline<=0'; return base

        base['baseline']   = float(T)
        base['mag_mean']   = float(np.mean(m))
        base['mag_std']    = float(np.std(m, ddof=1))
        base['err_median'] = float(np.median(e))

        snr, amp_int, passes = noise_check(m, e)
        base['snr'] = snr; base['amp_intrinsic'] = amp_int
        if not passes:
            base['error'] = 'snr<threshold'; return base

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            ls, freq, power = run_ls(t, m, e)

        i_raw = int(np.argmax(power))
        base['P_LS_raw'] = float(1.0 / freq[i_raw])

        good = alias_mask_vec(freq)
        if not np.any(good):
            base['error'] = 'all-alias'; return base

        peaks = top_k_peaks(freq[good], power[good], TOP_K_LS_PEAKS)
        if not peaks:
            base['error'] = 'no-peaks'; return base

        P_LS, power_LS   = peaks[0]
        base['P_LS']     = P_LS
        base['power_LS'] = power_LS
        base['alias_flag'] = not np.isclose(base['P_LS_raw'], P_LS, rtol=1e-3)

        try:
            fap = float(ls.false_alarm_probability(power_LS, method='baluev'))
        except Exception:
            fap = np.nan
        base['fap_LS'] = fap
        base['significant'] = (fap < FAP_THRESHOLD) if np.isfinite(fap) else False

        ref = refine_around_peaks(t, m, peaks)
        base.update(ref)

        P_final, agreement, n_agree = consensus_period(
            P_LS, ref['P_CE'], ref['P_PDM'])
        base['P_final']   = P_final
        base['agreement'] = agreement
        base['n_agree']   = n_agree
        base['valid']     = True
        return base

    except Exception as ex:
        base['error'] = f"{type(ex).__name__}: {ex}\n{traceback.format_exc()}"
        return base


# =============================================================================
# DISPATCH
# =============================================================================

def process_all(stars, n_workers, executor_cls):
    n_total = len(stars)
    print(f"\nProcesando {n_total} estrellas con {n_workers} workers "
          f"({executor_cls.__name__})...")

    results = [None] * n_total
    errors  = []
    t0 = pytime.time(); done = 0

    with executor_cls(max_workers=n_workers) as exe:
        futures = {exe.submit(_analyze_star, star): i
                   for i, star in enumerate(stars)}
        for fut in as_completed(futures):
            i = futures[fut]
            try:
                res = fut.result()
            except Exception as ex:
                res = {'source': stars[i][0], 'name': stars[i][1],
                       'valid': False, 'error': str(ex),
                       'n_obs': len(stars[i][2])}
            results[i] = res
            if res.get('error') and not res.get('valid'):
                errors.append((res['name'], res['error']))
            done += 1
            if done % 20 == 0 or done == n_total:
                elapsed = pytime.time() - t0
                rate = done / elapsed if elapsed > 0 else 0
                eta  = (n_total - done) / rate if rate > 0 else 0
                print(f"  {done:4d}/{n_total}  ({rate:.1f} est/s, ETA {eta:.0f}s)")

    reasons = {}
    for _, err in errors:
        key = err.split(':')[0].split('\n')[0] if err else 'unknown'
        reasons[key] = reasons.get(key, 0) + 1
    if reasons:
        print(f"  Descartadas/fallidas: {sum(reasons.values())}")
        for k, v in sorted(reasons.items(), key=lambda x: -x[1])[:5]:
            print(f"    {k}: {v}")

    return results


# =============================================================================
# CSV
# =============================================================================

CSV_COLS = [
    'source', 'name', 'valid', 'error', 'n_obs', 'baseline',
    'mag_mean', 'mag_std', 'err_median', 'snr', 'amp_intrinsic',
    'P_LS_raw', 'alias_flag', 'P_LS', 'power_LS', 'fap_LS', 'significant',
    'P_CE', 'CE_min', 'P_PDM', 'PDM_min',
    'P_final', 'agreement', 'n_agree',
]

def save_csv(results, path):
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLS)
        w.writeheader()
        for r in results:
            w.writerow({c: r.get(c, '') for c in CSV_COLS})
    print(f"  CSV escrito: {path}")


# =============================================================================
# PLOTS
# =============================================================================

def _render_star_page(args):
    record, t, m, e, order_key = args
    try:
        name = record['name']; src = record.get('source', '')

        fig    = Figure(figsize=(11, 8.5))
        canvas = FigureCanvasAgg(fig)  # noqa
        gs     = fig.add_gridspec(3, 2, hspace=0.45, wspace=0.28,
                                   left=0.08, right=0.96, top=0.90, bottom=0.08)

        P_LS, P_CE, P_PDM = record['P_LS'], record['P_CE'], record['P_PDM']
        P_final = record['P_final']; agr = record['agreement']
        fap_s = (f"{record['fap_LS']:.1e}"
                 if np.isfinite(record['fap_LS']) else "N/A")

        suptitle = (
            f"[{src}] {name}   N={record['n_obs']}   "
            f"baseline={record['baseline']:.0f} d   SNR={record['snr']:.1f}\n"
            f"P_LS={P_LS:.5f}  P_CE={P_CE:.5f}  P_PDM={P_PDM:.5f}   "
            f"-> P_final={P_final:.5f} d   [{agr}]   FAP={fap_s}"
        )
        fig.suptitle(suptitle, fontsize=9, y=0.98)

        ax = fig.add_subplot(gs[0, :])
        ax.errorbar(t, m, yerr=e, fmt='.', ms=3, lw=0.5, alpha=0.6, color='black')
        ax.invert_yaxis()
        ax.set_xlabel('Tiempo [d]'); ax.set_ylabel('mag')
        ax.set_title('Serie temporal', fontsize=10); ax.grid(alpha=0.3)

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            ls, freq, power = run_ls(t, m, e)

        ax = fig.add_subplot(gs[1, 0])
        ax.plot(1.0 / freq, power, lw=0.4, color='black')
        ax.axvline(P_LS,    color='tab:blue',  ls='--', lw=1, label=f'LS={P_LS:.4f}')
        ax.axvline(P_final, color='tab:green', ls='-',  lw=1, alpha=0.5, label=f'final={P_final:.4f}')
        ax.set_xscale('log')
        ax.set_xlabel('Periodo [d]'); ax.set_ylabel('Power LS')
        ax.set_title('Lomb-Scargle', fontsize=10)
        ax.legend(fontsize=7); ax.grid(alpha=0.3, which='both')

        hw_vis = 3 * REFINE_HALFWIDTH
        grid_vis = np.linspace(P_final * (1 - hw_vis),
                               P_final * (1 + hw_vis), REFINE_N_POINTS * 2)
        ce_vis  = conditional_entropy_batch(t, m, grid_vis)
        pdm_vis = pdm_batch(t, m, grid_vis)

        ax = fig.add_subplot(gs[1, 1])
        ax.plot(grid_vis, ce_vis, color='tab:orange', lw=0.8, label='CE')
        ax.axvline(P_CE,    color='tab:orange', ls='--', lw=1)
        ax.axvline(P_final, color='tab:green',  ls='-',  lw=1, alpha=0.4)
        ax.set_xlabel('Periodo [d]'); ax.set_ylabel('H(mag|fase)')
        ax.set_title(f'Conditional Entropy (CE_min={record["CE_min"]:.3f})', fontsize=10)
        ax.legend(fontsize=7); ax.grid(alpha=0.3)

        ax = fig.add_subplot(gs[2, 0])
        ax.plot(grid_vis, pdm_vis, color='tab:red', lw=0.8, label='PDM')
        ax.axvline(P_PDM,   color='tab:red',   ls='--', lw=1)
        ax.axvline(P_final, color='tab:green', ls='-',  lw=1, alpha=0.4)
        ax.set_xlabel('Periodo [d]'); ax.set_ylabel(r'$\Theta$')
        ax.set_title(f'PDM (Theta_min={record["PDM_min"]:.3f})', fontsize=10)
        ax.legend(fontsize=7); ax.grid(alpha=0.3)

        ax = fig.add_subplot(gs[2, 1])
        phase = ((t - t.min()) / P_final) % 1.0
        for off in (0, 1):
            ax.errorbar(phase + off, m, yerr=e, fmt='.', ms=3, lw=0.5,
                        alpha=0.6, color='steelblue')
        ax.invert_yaxis(); ax.set_xlim(0, 2)
        ax.set_xlabel('Fase'); ax.set_ylabel('mag')
        ax.set_title(f'Curva de fase (P = {P_final:.5f} d)', fontsize=10)
        ax.grid(alpha=0.3)

        buf = io.BytesIO()
        fig.savefig(buf, format='pdf', dpi=PLOT_DPI)
        buf.seek(0)
        return order_key, buf.read()

    except Exception as ex:
        fig    = Figure(figsize=(11, 8.5))
        canvas = FigureCanvasAgg(fig)  # noqa
        ax     = fig.add_subplot(111); ax.axis('off')
        ax.text(0.02, 0.98, f"ERROR en {record.get('name')}:\n"
                f"{type(ex).__name__}: {ex}\n{traceback.format_exc()}",
                fontsize=8, family='monospace', va='top', color='red',
                transform=ax.transAxes)
        buf = io.BytesIO(); fig.savefig(buf, format='pdf', dpi=PLOT_DPI)
        buf.seek(0)
        return order_key, buf.read()


def _render_summary(results, executor_label, sources):
    fig    = Figure(figsize=(11, 8.5))
    canvas = FigureCanvasAgg(fig)  # noqa
    axes   = fig.subplots(2, 2)
    fig.suptitle(f'Resumen multi-metodo   [fuentes: {", ".join(sources)}]',
                 fontsize=12, y=0.98)

    valid = [r for r in results if r['valid']]

    ax = axes[0, 0]
    srcs_unique = sorted({r.get('source', '?') for r in valid})
    bins = np.logspace(np.log10(0.05), np.log10(1500), 40)
    for src in srcs_unique:
        Ps = [r['P_final'] for r in valid
              if r.get('source') == src and np.isfinite(r['P_final'])]
        if Ps:
            ax.hist(Ps, bins=bins, alpha=0.5, label=f'{src} (n={len(Ps)})')
    ax.set_xscale('log')
    ax.set_xlabel('P_final [d]'); ax.set_ylabel('N estrellas')
    ax.set_title('Distribucion de P_final', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3, which='both')

    ax = axes[0, 1]
    snrs = [r['snr'] for r in results if np.isfinite(r.get('snr', np.nan))]
    if snrs:
        ax.hist(snrs, bins=40, color='coral', alpha=0.7, edgecolor='black')
    ax.axvline(SNR_MIN, color='red', ls='--', label=f'umbral={SNR_MIN}')
    ax.set_xlabel('SNR'); ax.set_ylabel('N')
    ax.set_title('Distribucion de SNR', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1, 0]
    agr_counts = {}
    for r in valid:
        k = r.get('agreement') or 'none'
        agr_counts[k] = agr_counts.get(k, 0) + 1
    if agr_counts:
        keys = sorted(agr_counts.keys(), key=lambda k: -agr_counts[k])
        ax.bar(range(len(keys)), [agr_counts[k] for k in keys],
               color='slateblue', alpha=0.8)
        ax.set_xticks(range(len(keys)))
        ax.set_xticklabels(keys, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('N estrellas')
    ax.set_title('Consenso entre metodos', fontsize=10)
    ax.grid(alpha=0.3, axis='y')

    ax = axes[1, 1]; ax.axis('off')
    lines = ["Conteo por fuente", "-" * 40]
    for src in srcs_unique:
        rs = [r for r in results if r.get('source') == src]
        vs = [r for r in rs if r['valid']]
        n3 = sum(1 for r in vs if r['n_agree'] == 3)
        lines += [
            f"{src}:",
            f"  total        : {len(rs)}",
            f"  validas      : {len(vs)}",
            f"  consenso 3/3 : {n3}",
            "",
        ]
    lines += [
        "Hiperparametros",
        "-" * 40,
        f"period_min    = {PERIOD_MIN} d",
        f"period_max    = T x {PERIOD_MAX_FACTOR:.3f}",
        f"min_obs       = {MIN_OBSERVATIONS}",
        f"SNR_min       = {SNR_MIN}",
        f"FAP_threshold = {FAP_THRESHOLD}",
        f"TOP_K_LS      = {TOP_K_LS_PEAKS}",
        f"REFINE +-/N   = {REFINE_HALFWIDTH}/{REFINE_N_POINTS}",
        f"CE bins       = ({CE_PHASE_BINS},{CE_MAG_BINS})",
        f"PDM bins      = {PDM_N_BINS}",
        f"CONSENSUS_RTOL= {CONSENSUS_RTOL}",
        f"Executor      = {executor_label}",
    ]
    ax.text(0.02, 0.98, "\n".join(lines), fontsize=9,
            family='monospace', va='top', transform=ax.transAxes)

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    buf = io.BytesIO()
    fig.savefig(buf, format='pdf', dpi=PLOT_DPI)
    buf.seek(0)
    return buf.read()


def generate_pdf(stars_by_key, results, path, n_workers, executor_cls,
                 executor_label, sources):
    try:
        from pypdf import PdfWriter, PdfReader
    except ImportError:
        print("  pypdf no instalado (pip install pypdf). Solo resumen.")
        with open(path, 'wb') as f:
            f.write(_render_summary(results, executor_label, sources))
        return

    valid = [r for r in results if r['valid']]
    valid.sort(key=lambda r: (r['n_agree'], r['snr']), reverse=True)
    selected = valid[:N_TOP_PLOTS] if (N_TOP_PLOTS and N_TOP_PLOTS > 0) else valid

    print(f"  Generando PDF (~{1 + len(selected)} paginas)...")

    render_tasks = []
    for k, r in enumerate(selected):
        key = (r['source'], r['name'])
        if key not in stars_by_key:
            continue
        t, m, e = stars_by_key[key]
        render_tasks.append((r, t, m, e, k))

    page_bytes = {}
    t0, done, total = pytime.time(), 0, len(render_tasks)
    with executor_cls(max_workers=n_workers) as exe:
        futures = {exe.submit(_render_star_page, task): task[4]
                   for task in render_tasks}
        for fut in as_completed(futures):
            try:
                key, b = fut.result()
                page_bytes[key] = b
            except Exception as ex:
                print(f"    render fallido: {ex}")
            done += 1
            if done % 10 == 0 or done == total:
                elapsed = pytime.time() - t0
                rate = done / elapsed if elapsed > 0 else 0
                eta  = (total - done) / rate if rate > 0 else 0
                print(f"    renders {done}/{total}  ({rate:.1f} pag/s, ETA {eta:.0f}s)")

    print("  Ensamblando PDF...")
    writer = PdfWriter()
    summary = _render_summary(results, executor_label, sources)
    writer.add_page(PdfReader(io.BytesIO(summary)).pages[0])
    for key in sorted(page_bytes.keys()):
        writer.add_page(PdfReader(io.BytesIO(page_bytes[key])).pages[0])
    with open(path, 'wb') as f:
        writer.write(f)
    print(f"  PDF guardado: {path}")


# =============================================================================
# MAIN
# =============================================================================

def main():
    n_cpu     = mp.cpu_count()
    n_workers = N_WORKERS if N_WORKERS else n_cpu
    executor_cls, executor_label = _select_executor_class()

    OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

    print("=" * 65)
    print("Pipeline multi-metodo (LS + CE + PDM)")
    print("=" * 65)
    print(f"  Data dir   : {DATA_DIR}")
    print(f"  Salida     : {OUTPUT_DIR.resolve()}")
    print(f"  Workers    : {n_workers} / {n_cpu}")
    print(f"  Executor   : {executor_label}")

    sources = resolve_sources()
    print(f"\n  Fuente(s) seleccionada(s): {', '.join(sources)}")

    if executor_cls is ThreadPoolExecutor and EXECUTOR_MODE == 'auto':
        print("  [INFO] Threads (notebook). Para mas velocidad: "
              "python periods_pipeline.py")

    t0 = pytime.time()
    stars = load_selected(sources)
    print(f"\n  Total estrellas a procesar: {len(stars)}")

    results = process_all(stars, n_workers, executor_cls)

    print("\n" + "=" * 65)
    print("Resumen")
    print("=" * 65)
    for src in sorted({s[0] for s in stars}):
        rs    = [r for r in results if r.get('source') == src]
        valid = [r for r in rs if r['valid']]
        sig   = [r for r in valid if r.get('significant')]
        c3    = [r for r in valid if r['n_agree'] == 3]
        print(f"  {src:12s}: total {len(rs):4d}  validas {len(valid):4d}  "
              f"signif {len(sig):4d}  consenso 3/3 {len(c3):4d}")

# 1. Agrupar las estrellas originales usando su fuente y nombre como llave
    stars_by_key = {(s[0], s[1]): (s[2], s[3], s[4]) for s in stars}
    
    # 2. Identificar todas las fuentes únicas que realmente se procesaron (ej: 'h5', 'fits_B278')
    sources_present = sorted({r.get('source', 'unknown') for r in results if r})

    # 3. Iterar sobre cada fuente y generar sus propios archivos
    for src in sources_present:
        print(f"\n" + "-"*40)
        print(f"Guardando resultados separados para: {src}")
        print("-" * 40)
        
        # Filtrar la lista gigante para quedarnos solo con los resultados de ESTA fuente
        src_results = [r for r in results if r and r.get('source') == src]
        
        # Nombres dinámicos: agregamos el nombre de la fuente al final del archivo
        csv_path = OUTPUT_DIR / f'periods_results_{src}.csv'
        pdf_path = OUTPUT_DIR / f'periods_report_{src}.pdf'
        
        # Ejecutar el guardado con la lista filtrada
        save_csv(src_results, csv_path)
        
        print(f"  Generando PDF para {src}...")
        # Nota: Le pasamos [src] como argumento final para que el reporte sepa de qué trata
        generate_pdf(stars_by_key, src_results, pdf_path,
                     n_workers, executor_cls, executor_label, [src])

    print(f"\nListo en {pytime.time() - t0:.1f} s.")


if __name__ == '__main__':
    mp.freeze_support()
    main()

Pipeline multi-metodo (LS + CE + PDM)
  Data dir   : C:\Users\santi\Downloads\Lab 5
  Salida     : C:\Users\santi\Downloads\Experimental\periods_output
  Workers    : 16 / 16
  Executor   : ThreadPoolExecutor (notebook)

Seleccion de fuente de datos
  [1] HDF5  : singlemode_variable_stars_sample.h5       OK
  [2] FITS  : VVV_Sample.fits                           OK
  [3] Ambos

  Fuente(s) seleccionada(s): h5, fits
  [INFO] Threads (notebook). Para mas velocidad: python periods_pipeline.py

Leyendo HDF5: C:\Users\santi\Downloads\Lab 5\singlemode_variable_stars_sample.h5
  140 estrellas.

Leyendo FITS: C:\Users\santi\Downloads\Lab 5\VVV_Sample.fits
  590 estrellas (B278 + B279).

  Total estrellas a procesar: 730

Procesando 730 estrellas con 16 workers (ThreadPoolExecutor)...
    20/730  (0.5 est/s, ETA 1395s)
    40/730  (0.6 est/s, ETA 1120s)
    60/730  (0.7 est/s, ETA 1009s)
    80/730  (0.7 est/s, ETA 966s)
   100/730  (0.7 est/s, ETA 852s)
   120/730  (0.7 est/s, ETA 855s)
   140